In [ ]:
# Install the baseline and shared evaluation packages.
!pip install -q -U "transformers>=4.48,<5" accelerate langchain-huggingface langchain-core pydantic python-dotenv requests huggingface_hub pandas rapidfuzz sacrebleu bert-score==0.3.13


In [ ]:
# Import the original baseline pipeline and evaluation dependencies.
import json, re, unicodedata, torch, pandas as pd
from pathlib import Path
from collections import Counter
from pydantic import BaseModel
from typing import List
from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda


In [ ]:
# Load the same 200-question test set from the requested GitHub Data folder.
import requests
TEST_URL = "https://raw.githubusercontent.com/RamijWasithRahat/Bangla-Agriculture-Chatbot/main/Data/Bangla_Agriculture_QA_Test_200.json"
class QNAPair(BaseModel):
    id: int
    question: str
    reference_answer: str
class AgriData(BaseModel):
    qa_pairs: List[QNAPair]
data = requests.get(TEST_URL, timeout=60).json()
agri_data = AgriData(**data)
print("Test examples:", len(agri_data.qa_pairs))


In [ ]:
# Log in to Hugging Face before loading the gated Llama model.
from huggingface_hub import notebook_login
notebook_login()


In [ ]:
# Load the original Llama 3.2 1B Instruct baseline model.
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (torch.float16 if torch.cuda.is_available() else torch.float32)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype, device_map="auto" if torch.cuda.is_available() else None, low_cpu_mem_usage=True)
terminators = [tokenizer.eos_token_id]
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
if isinstance(eot_id, int) and eot_id >= 0 and eot_id not in terminators:
    terminators.append(eot_id)


In [ ]:
# Keep the original deterministic Hugging Face generation pipeline unchanged.
pipe = pipeline(
    "text-generation", model=model, tokenizer=tokenizer,
    max_new_tokens=100, do_sample=False, return_full_text=False,
    pad_token_id=tokenizer.pad_token_id, eos_token_id=terminators,
)
llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
# Define the original zero-shot prompt.
zero_shot_prompt = PromptTemplate(
    template="""নিচের প্রশ্নটির উত্তর বাংলায় দিন।\n\nপ্রশ্নঃ {question}\nউত্তরঃ""",
    input_variables=["question"],
)


In [ ]:
# Define the original one-shot prompt.
one_shot_prompt = PromptTemplate(
    template="""নিচের উদাহরণটি লক্ষ্য করুন এবং একইভাবে বাংলায় উত্তর দিন।\n\nপ্রশ্নঃ টমেটো গাছে পার্শ্বকুশি ও মরা পাতা ছাঁটাই করার সুবিধা কী?\nউত্তরঃ পার্শ্বকুশি ও মরা পাতা ছাঁটাই করলে গাছে রোগ-বালাই ও\nপোকার আক্রমণ কমে এবং ফলের আকার বড় হয়।\n\nপ্রশ্নঃ {question}\nউত্তরঃ""",
    input_variables=["question"],
)


In [ ]:
# Define the original few-shot prompt.
few_shot_prompt = PromptTemplate(
    template="""নিচের উদাহরণগুলো লক্ষ্য করুন এবং একইভাবে বাংলায় উত্তর দিন।\n\nপ্রশ্নঃ টমেটোর চারা রোপণের সঠিক দূরত্ব ও বয়স কত?\nউত্তরঃ ২৫-৩০ দিন বয়সের টমেটোর চারা প্রতিটি বেডে দুটি\nসারিতে ৬০×৪০ সেমি দূরত্বে রোপণ করতে হয়।\n\nপ্রশ্নঃ টমেটো চাষে হেক্টরপ্রতি কী পরিমাণ গোবর সার প্রয়োগ করতে হয়?\nউত্তরঃ টমেটোর ভালো ফলনের জন্য হেক্টরপ্রতি ৮ থেকে ১২ টন\nপচা গোবর সার প্রয়োগ করতে হয়।\n\nপ্রশ্নঃ টমেটোর জমিতে ইউরিয়া এবং এমপি সার কখন ও কীভাবে দিতে হয়?\nউত্তরঃ পার্শ্বকুশি ছাঁটাইয়ের পর চারা লাগানোর ৩য় ও ৫ম\nসপ্তাহে রিং পদ্ধতিতে দুই কিস্তিতে এই সারগুলো দিতে হয়।\n\nপ্রশ্নঃ টমেটো গাছে পার্শ্বকুশি ও মরা পাতা ছাঁটাই করার সুবিধা কী?\nউত্তরঃ পার্শ্বকুশি ও মরা পাতা ছাঁটাই করলে গাছে রোগ-বালাই ও\nপোকার আক্রমণ কমে এবং ফলের আকার বড় হয়।\n\nপ্রশ্নঃ টমেটো গাছকে প্রবল বাতাস থেকে রক্ষা করতে কী ব্যবস্থা নেওয়া হয়?\nউত্তরঃ টমেটো গাছ যাতে নুয়ে না পড়ে সেজন্য বাঁশের তৈরি কাঠি,\nকঞ্চি বা ডাল দিয়ে ‘এ’ আকৃতির ঠেকনা দেওয়া হয়।\n\nপ্রশ্নঃ {question}\nউত্তরঃ""",
    input_variables=["question"],
)


In [ ]:
# Define the original chain-of-thought-style prompt.
cot_prompt = PromptTemplate(
    template="""আপনি একজন কৃষি বিশেষজ্ঞ।\n\nপ্রশ্নটি বুঝে প্রয়োজনীয় তথ্য বিবেচনা করুন।\nসবশেষে শুধু সংক্ষিপ্ত ও সঠিক চূড়ান্ত উত্তর বাংলায় দিন।\n\nপ্রশ্নঃ {question}\n\nচূড়ান্ত উত্তরঃ""",
    input_variables=["question"],
)


In [ ]:
# Define the original instruction prompt.
instruction_prompt = PromptTemplate(
    template="""আপনি একজন কৃষি সহায়ক বাংলা প্রশ্নোত্তর সহকারী।\n\nনির্দেশনা:\n- শুধুমাত্র বাংলায় উত্তর দিন\n- সংক্ষিপ্ত ও নির্ভুল উত্তর দিন\n- অতিরিক্ত ব্যাখ্যা যোগ করবেন না\n- প্রশ্ন অনুযায়ী সরাসরি উত্তর দিন\n\nপ্রশ্নঃ {question}\nউত্তরঃ""",
    input_variables=["question"],
)


In [ ]:
# Register all five original baseline prompting methods.
PROMPTS = {
    "zero_shot": zero_shot_prompt,
    "one_shot": one_shot_prompt,
    "few_shot": few_shot_prompt,
    "chain_of_thought": cot_prompt,
    "instruction": instruction_prompt,
}


In [ ]:
# Build the same chat-template LangChain flow for any selected prompt.
SYSTEM_TEXT = "আপনি একজন কৃষি বিষয়ক প্রশ্নোত্তর সহকারী। আপনাকে বাংলায় সংক্ষিপ্ত ও সঠিক উত্তর দিতে হবে।"
def make_chain(prompt_template):
    def format_llama_prompt(inputs):
        user_prompt = prompt_template.format(question=inputs["question"])
        messages = [{"role": "system", "content": SYSTEM_TEXT}, {"role": "user", "content": user_prompt}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return RunnableLambda(format_llama_prompt) | llm | StrOutputParser()


In [ ]:
# Generate and retain separate predictions for every prompting method.
BASELINE_OUT = Path("./baseline_results")
BASELINE_OUT.mkdir(exist_ok=True)
method_predictions = {}
for method, prompt_template in PROMPTS.items():
    chain = make_chain(prompt_template)
    rows = []
    for pair in agri_data.qa_pairs:
        prediction = chain.invoke({"question": pair.question}).strip()
        token_count = len(tokenizer(prediction, add_special_tokens=False)["input_ids"])
        rows.append({"id": pair.id, "question": pair.question, "gold": pair.reference_answer, "prediction": prediction, "truncated": token_count >= 100})
    method_predictions[method] = pd.DataFrame(rows)
    print(method, "completed:", len(rows))


In [ ]:
# Use the RAG notebook text normalization exactly for every evaluation.
BN_TO_EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
def normalize(text):
    text = unicodedata.normalize("NFKC", str(text)).translate(BN_TO_EN).lower()
    text = re.sub(r"[^\u0980-\u09FFA-Za-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()
def tokens(text):
    return normalize(text).split()


In [ ]:
# Use the same Exact Match and Token F1 definitions .
def exact_match(pred, gold):
    return float(normalize(pred) == normalize(gold))
def token_f1(pred, gold):
    p, g = tokens(pred), tokens(gold)
    if not p or not g:
        return 0.0
    overlap = sum((Counter(p) & Counter(g)).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / len(p), overlap / len(g)
    return 2 * precision * recall / (precision + recall)


In [ ]:
# Use the same ROUGE-1 and ROUGE-2 definitions .
def rouge_n(pred, gold, n):
    p, g = tokens(pred), tokens(gold)
    if len(p) < n or len(g) < n:
        return 0.0
    pg = Counter(tuple(p[i:i+n]) for i in range(len(p)-n+1))
    gg = Counter(tuple(g[i:i+n]) for i in range(len(g)-n+1))
    overlap = sum((pg & gg).values())
    if overlap == 0:
        return 0.0
    precision, recall = overlap / sum(pg.values()), overlap / sum(gg.values())
    return 2 * precision * recall / (precision + recall)


In [ ]:
# Use the same ROUGE-L definition .
def rouge_l(pred, gold):
    p, g = tokens(pred), tokens(gold)
    if not p or not g:
        return 0.0
    dp = [0] * (len(g) + 1)
    for x in p:
        new = [0]
        for j, y in enumerate(g, 1):
            new.append(dp[j-1] + 1 if x == y else max(dp[j], new[-1]))
        dp = new
    lcs = dp[-1]
    precision, recall = lcs / len(p), lcs / len(g)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


In [ ]:
# Evaluate a prediction table with the exact RAG metric configuration.
def evaluate_rag_metrics(df):
    df = df.copy().fillna("")
    df["Exact Match"] = [exact_match(p, g) for p, g in zip(df["prediction"], df["gold"])]
    df["Fuzzy Match"] = [fuzz.token_set_ratio(normalize(p), normalize(g)) / 100 for p, g in zip(df["prediction"], df["gold"])]
    df["Token F1"] = [token_f1(p, g) for p, g in zip(df["prediction"], df["gold"])]
    df["ROUGE-1"] = [rouge_n(p, g, 1) for p, g in zip(df["prediction"], df["gold"])]
    df["ROUGE-2"] = [rouge_n(p, g, 2) for p, g in zip(df["prediction"], df["gold"])]
    df["ROUGE-L"] = [rouge_l(p, g) for p, g in zip(df["prediction"], df["gold"])]
    bleu = BLEU(tokenize="none", smooth_method="exp", effective_order=True)
    pred_texts, gold_texts = [" ".join(tokens(x)) for x in df["prediction"]], [" ".join(tokens(x)) for x in df["gold"]]
    corpus_bleu = bleu.corpus_score(pred_texts, [gold_texts]).score / 100
    P, R, F1 = bert_score(df["prediction"].astype(str).tolist(), df["gold"].astype(str).tolist(), model_type="bert-base-multilingual-cased", batch_size=4, device="cpu", idf=False, rescale_with_baseline=False, verbose=True)
    df["BERT Precision"], df["BERT Recall"], df["BERT F1"] = P.cpu().numpy(), R.cpu().numpy(), F1.cpu().numpy()
    values = [df["Exact Match"].mean(), df["Fuzzy Match"].mean(), corpus_bleu, df["ROUGE-1"].mean(), df["ROUGE-2"].mean(), df["ROUGE-L"].mean(), df["Token F1"].mean(), df["BERT Precision"].mean(), df["BERT Recall"].mean(), df["BERT F1"].mean(), df["truncated"].astype(str).str.lower().eq("true").sum()]
    names = ["Exact Match", "Fuzzy Match", "Corpus BLEU", "ROUGE-1", "ROUGE-2", "ROUGE-L", "Token F1", "BERT Precision", "BERT Recall", "BERT F1", "Truncated Outputs"]
    return df, pd.DataFrame({"metric": names, "score": values})


In [ ]:
# Save separate prediction and result CSV files for every baseline method.
all_results = []
for method, pred_df in method_predictions.items():
    scored_df, result_df = evaluate_rag_metrics(pred_df)
    scored_df.to_csv(BASELINE_OUT / f"{method}_predictions.csv", index=False, encoding="utf-8-sig")
    result_df.to_csv(BASELINE_OUT / f"{method}_result.csv", index=False, encoding="utf-8-sig")
    temp = result_df.copy()
    temp.insert(0, "method", method)
    all_results.append(temp)
    print("Saved:", method)


In [ ]:
# Save one combined CSV to compare all baseline prompting methods easily.
baseline_comparison = pd.concat(all_results, ignore_index=True)
baseline_comparison.to_csv(BASELINE_OUT / "baseline_all_methods_result.csv", index=False, encoding="utf-8-sig")
display(baseline_comparison.pivot(index="metric", columns="method", values="score"))
print("Saved to:", BASELINE_OUT.resolve())
